In [7]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [8]:
from statsmodels.tsa.seasonal import seasonal_decompose

import torch
import numpy as np
import random
import os

# Set seeds for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)

# CUDA deterministic settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [9]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import transforms

from ilipy import Session
from ilipy.database import DistanceCorrelation
from ilipy import  Session

# from rnd.utils.modeling import get_model
# from rnd.utils.transform import resize_to_224
from data_utils import (
    set_ili_run,
    extract_arm_angles,
    create_arm_array,
)

ModuleNotFoundError: No module named 'data_utils'

In [ ]:
run_number = 6
root_dir = f"./data/fhr{run_number}"
os.makedirs(root_dir, exist_ok=True)
_, _, inspection_id, env, _, start_dist, end_dist = set_ili_run(run_number)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NameError: name 'set_ili_run' is not defined

DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: 

In [4]:
session = Session(environment="research")
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)

PySettings.cpp(46): ilipy: Build 0.17.2.10867 5d2ea142f2 release-ili-0.17 'Tue Jul 22 21:50:18 2025'
PySettings.cpp(47): Paths modulePath: /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy, userHomePath: /home/zmirikha, identityPath: /home/zmirikha/.aws/ilipy
Settings.cpp(246): Settings: loading settings from /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy/Data/Environments/defaults.json
Settings.cpp(246): Settings: loading settings from /home/zmirikha/ilipy/lib/python3.11/site-packages/ilipy/Data/Environments/research.json
PyAuthenticator.cpp(239): Attempting an IAM role login as no username and password or environment variables were provided.
CognitoAuthenticator.cpp(158): Using Cognito Credentials
CognitoAuthenticator.cpp(842): Login Successful.
CognitoAuthenticator.cpp(842): Login Successful.
CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji
CognitoAuthenticator.cpp(883): Logged in as username: zahra.mirikharaji
DatabaseWebSocket.cpp(168): WebSoc

NameError: name 'inspection_id' is not defined

In [ ]:
class PreprocessingModel(nn.Module):
    def __init__(self, model, transform):
        super().__init__()
        self.model = model
        self.transform = transform

    def forward(self, input_np_image):
        # Apply transform to NumPy image (expects single image, not batch)
        x = self.transform(input_np_image)
        x = x.unsqueeze(0)  # Add batch dimension
        return self.model(x)

In [ ]:
def load_model(model_name, dense_units, dropout, model_path):
    """
    Load a PyTorch model with weights and return it along with the transform.

    Returns:
        model (torch.nn.Module): The loaded model ready for inference.
        transform (callable): The image preprocessing pipeline.
        weights: The weight metadata used for normalization.
    """
    model, weights = get_model(model_name, dense_units, dropout)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model, weights

    # transform = transforms.Compose([
    #     transforms.Lambda(resize_to_224),
    #     transforms.ToTensor(),
    #     transforms.Lambda(lambda x: x.expand(3, -1, -1)),
    #     transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std),
    # ])

    # wrapped_model = PreprocessingModel(model, transform)
    #return wrapped_model

In [ ]:
model, weights = load_model("resnet", 128, 0.5, "models/dent_models/resnet_final_40.pth")

In [ ]:
transform = transforms.Compose(
    [
        transforms.Lambda(resize_to_224), # should output (C, H, W)
        transforms.Lambda(lambda x: torch.from_numpy(x).float()),
        transforms.Normalize(
            mean=weights.transforms().mean, std=weights.transforms().std
        ),
    ]
)

In [ ]:
def gen_image(img_start, length= 0.3, show_im=True, normalize="range"):
    img_range = (img_start, img_start + length)
    tick_sampling_interval = 10

    arm_angles = extract_arm_angles(session, dist_corr, img_range, tick_sampling_interval=tick_sampling_interval)
    num_tick_samples = int(length * 10000 / tick_sampling_interval)
    img = create_arm_array(
    arm_angles,
    num_tracks=22,
    num_tick_samples=num_tick_samples,
    normalize=''#normalize,
)
    img_s = create_arm_array(
    arm_angles,
    num_tracks=22,
    num_tick_samples=num_tick_samples,
    normalize='std'#normalize,
)
    if show_im: 
        plt.imshow(
            img_s.T, cmap="inferno", origin="lower", aspect="auto",
            interpolation="nearest",
        )

        plt.axis("off")
        plt.show()
    return img

In [ ]:
patch_length = 0.3
stride = 0.25
ranges = [(start, start + patch_length) for start in np.arange(start_dist, end_dist - patch_length + stride, stride)]

In [ ]:
len(ranges)

In [ ]:
img_std_dir = os.path.join(root_dir, "img_std")
img_patch_dir = os.path.join(root_dir, "img_patch")
os.makedirs(img_std_dir, exist_ok=True)
os.makedirs(img_patch_dir, exist_ok=True)
for i, (img_start, img_end) in enumerate(ranges):
        if i % 200 == 0:
            print(f"Processing image {i}/{len(ranges)}")
        vd = (img_start + img_end) / 2 #147828.385 -0.15#
        # img_std = gen_image(vd, show_im=False, normalize="std")
        img_patch = gen_image(vd, show_im=False, normalize="")
        # np.save(os.path.join(img_std_dir, f"img_std_{vd*1000:.0f}.npy"), img_std)
        np.save(os.path.join(img_patch_dir, f"img_patch_{vd*1000:.0f}.npy"), img_patch)

In [ ]:
from model import DentPerTrackModel
from preprocess import Normalizer, build_inputs_linear, decompose, to_device
# ckpt_path = r"D:\ILIRepo\ILIWorkflows\workflows\dent_detection\time_series\checkpoints\20250816_231408_deltas1_normperseq_zscore_decompose0_trendonly0\best_model.pth"
ckpt_path = r"D:\ILIRepo\ILIWorkflows\workflows\dent_detection\time_series\checkpoints\20250817_211101_deltas0_normtrain_zscore_decompose0_trendonly0_0.005\best_model.pth"
def load_model_and_normalizer_from_ckpt(ckpt_path: str, device: torch.device):
    """
    Rebuild model & normalizer from a training checkpoint and load weights/stats.
    """
    ckpt = torch.load(ckpt_path, map_location=device)

    # Pull architecture flags from checkpoint to ensure a perfect match
    ckpt_args = ckpt.get("args", {})
    use_deltas = ckpt.get("use_deltas", ckpt_args.get("use_deltas", False))
    embed_dim   = ckpt_args.get("embed_dim", 32)
    use_xattn   = ckpt_args.get("use_xattn", False)
    xattn_heads = ckpt_args.get("xattn_heads", 1)
    xattn_layers= ckpt_args.get("xattn_layers", 1)

    per_track_in_ch = 2 if use_deltas else 1
    model = DentPerTrackModel(
        per_track_in_ch=per_track_in_ch,
        embed_dim=embed_dim,
        use_xattn=use_xattn,
        xattn_heads=xattn_heads,
        xattn_layers=xattn_layers,
    ).to(device)
    model.load_state_dict(ckpt["model_state"])
    norm_mode = ckpt.get("norm_mode", ckpt_args.get("norm_mode", "train_zscore"))
    print(f"Loading model with norm_mode='{norm_mode}'")
    normalizer = Normalizer(mode=norm_mode).to(device)
    if norm_mode in ("train_zscore", "center_perseq_scale_train"):
        mean_train = ckpt.get("mean_train", None)
        std_train  = ckpt.get("std_train", None)
        if mean_train is None or std_train is None:
            raise RuntimeError(
                f"Checkpoint lacks mean/std for norm_mode='{norm_mode}'."
            )
        normalizer.mean_train.resize_(mean_train.shape).copy_(mean_train.to(device))
        normalizer.std_train.resize_(std_train.shape).copy_(std_train.to(device))
        normalizer.fitted = True
    else:
        normalizer.fitted = True

    return model, normalizer, use_deltas, ckpt_args

def load_model(ckpt_path):
    """
    Load a PyTorch model with weights and return it along with the transform.

    Returns:
        model (torch.nn.Module): The loaded model ready for inference.
        transform (callable): The image preprocessing pipeline.
        weights: The weight metadata used for normalization.
    """
    model, normalizer, use_deltas_from_ckpt, ckpt_args = load_model_and_normalizer_from_ckpt(
        ckpt_path, device
    )
    model.eval()
    return model, normalizer, use_deltas_from_ckpt, ckpt_args
    # model, weights = get_model(model_name, dense_units, dropout)
    # model.load_state_dict(torch.load(model_path))
    # model.eval()
    # return model, weights

ModuleNotFoundError: No module named 'model'

DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy
DbHealthCheck.cpp(146): Database service health check complete: Healthy


In [ ]:
model, normalizer, use_deltas_from_ckpt, ckpt_args = load_model(ckpt_path)

In [ ]:
import os, numpy as np, torch

# --- device & modules ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
if hasattr(normalizer, "to"): normalizer.to(device)

# auto-match deltas with model training
auto_use_deltas = (getattr(model, "per_track_in_ch", 1) == 2)

def run_slice(x20: torch.Tensor) -> torch.Tensor:
    """
    x20: (B, 20, T) on the SAME device as model/normalizer
    returns: probs (B, 20)
    """
    X_feat = build_inputs_linear(x20, use_deltas=auto_use_deltas)  # (B, 20|40, T)
    exp_cin = 20 * getattr(model, "per_track_in_ch", 1)
    assert X_feat.shape[1] == exp_cin, f"Cin={X_feat.shape[1]} but expected {exp_cin}"
    X_feat = normalizer(X_feat)
    logits = model.forward_with_features(X_feat)                   # (B, 20)
    return torch.sigmoid(logits)

# optional: choose how to fuse overlap tracks 2..19
combine = "avg"   # or "max"
global_thresh = 0.5
high_track_thresh = 0.9

preds, probs, preds_list = [], [], {}

# ---- temporarily disable deterministic algos to avoid cuBLAS error ----
was_det = torch.are_deterministic_algorithms_enabled()
if was_det:
    torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True  # safe for eval

try:
    with torch.no_grad():
        batch_size = 64
        for batch_start in range(0, len(ranges), batch_size):
            print(f"Processing batch {batch_start // batch_size + 1}")
            batch_ranges = ranges[batch_start:batch_start + batch_size]

            # load a batch of (22, T)
            imgs_22, view_dists = [], []
            for img_start, img_end in batch_ranges:
                vd = (img_start + img_end) / 2.0
                arr = np.load(os.path.join(img_patch_dir, f"img_patch_{vd*1000:.0f}.npy"))[:22, :]
                arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
                imgs_22.append(torch.from_numpy(arr).float())
                view_dists.append(vd)

            imgs_22 = torch.stack(imgs_22, dim=0).to(device)   # (B, 22, T)
            assert imgs_22.shape[1] == 22, f"Expected 22 tracks, got {imgs_22.shape}"

            # two passes of 20 tracks
            img_a = imgs_22[:, :20, :]   # (B, 20, T) -> global 0..19
            img_b = imgs_22[:,  2:, :]   # (B, 20, T) -> global 2..21

            p_a = run_slice(img_a)       # (B, 20)
            p_b = run_slice(img_b)       # (B, 20)

            # fuse into (B, 22)
            B = imgs_22.shape[0]
            probs_22 = torch.empty(B, 22, device=device)
            probs_22[:, 0:20] = p_a
            if combine == "avg":
                probs_22[:, 2:20] = 0.5 * (probs_22[:, 2:20] + p_b[:, 0:18])  # overlap 2..19
            else:
                probs_22[:, 2:20] = torch.maximum(probs_22[:, 2:20], p_b[:, 0:18])
            probs_22[:, 20:22] = p_b[:, 18:20]                                  # 20..21

            p_global   = probs_22.max(dim=1).values
            pred_track = probs_22.argmax(dim=1)

            preds.extend(pred_track.detach().cpu().tolist())
            probs.extend(p_global.detach().cpu().tolist())

            p22_np = probs_22.detach().cpu().numpy()
            pg_np  = p_global.detach().cpu().numpy()
            for vd, vec, pg in zip(view_dists, p22_np, pg_np):
                if pg >= global_thresh:
                    hi = np.where(vec > high_track_thresh)[0].tolist()
                    if hi: preds_list[vd] = hi
finally:
    # restore determinism flags
    if was_det:
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
img_std_dir = os.path.join(root_dir, "img_std")
img_patch_dir = os.path.join(root_dir, "img_patch")
batch_size = 64
preds = []
probs = []
preds_list = {}

with torch.no_grad():
    model.eval()
    model.to(device)
    for batch_start in range(0, len(ranges), batch_size):
        print(f"Processing batch {batch_start // batch_size + 1}")
        batch_ranges = ranges[batch_start:batch_start + batch_size]
        imgs = []
        for img_start, img_end in batch_ranges:
            vd = (img_start + img_end) / 2
            img_std = np.load(os.path.join(img_std_dir, f"img_std_{vd*1000:.0f}.npy"))[:22, :]
            img_patch = np.load(os.path.join(img_patch_dir, f"img_patch_{vd*1000:.0f}.npy"))[:22, :]
            img_patch = np.nan_to_num(img_patch, nan=0.0, posinf=0.0, neginf=0.0)
            # img_patch_trend = seasonal_decompose(img_patch.T, model='additive', period=30, extrapolate_trend=2).trend.T
            # img = np.stack((img_patch, img_patch_trend, img_std), axis=0)  # shape: (3, H, W)
            # img = transform(img)
            imgs.append(img_patch)
        imgs = torch.stack(imgs, dim=0).to(device)  # shape: (batch_size, 3, H, W)
        raw_outputs = model(imgs)
        preds.extend(raw_outputs.argmax(dim=1).cpu().numpy())
        probs.extend(raw_outputs.sigmoid().max(dim=1).values.cpu().numpy())
        for br, raw_output in zip(batch_ranges, raw_outputs.sigmoid().cpu().numpy()):
            if raw_output.argmax() == 0:
                continue
            else:
                view_dist = (br[0] + br[1]) / 2
                preds_list[view_dist] = list((raw_output > 0.9).nonzero()[0])

In [ ]:
#preds = [pred.cpu() for pred in preds]
np.save(os.path.join(root_dir, "preds4.npy"), np.array(preds))
np.save(os.path.join(root_dir, "probs4.npy"), np.array(probs))


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(probs, bins=50, color='orange', alpha=0.7)
plt.xlabel('Confidence Probability')
plt.ylabel('Frequency')
plt.title('Histogram of Confidence Probabilities')
plt.grid(axis='y', alpha=0.3)
plt.show()

thresholds = [0.5, 0.7, 0.8, 0.9]
for t in thresholds:
    count = sum(p > t for p in probs)
    print(f"Number of items in probs > {t}: {count}")

In [ ]:
len(preds), len(probs)

In [ ]:
# preds = np.load(os.path.join(root_dir, "preds4.npy"))
# probs = np.load(os.path.join(root_dir, "probs4.npy"))

In [ ]:
len(preds), len(probs), len (ranges)

In [ ]:
# Plot predictions
plt.style.use('dark_background')
plt.figure(figsize=(10, 6))

# Create x-axis points
patch_length = 0.3
x_points = [(img_start + img_end) / 2 for (img_start, img_end) in ranges]

# Plot class predictions (0 or 1)
plt.subplot(2, 1, 1)
plt.step(x_points, preds, where='mid', linewidth=2)
plt.ylabel('Predicted Class')
plt.title('Model Predictions vs Distance')
plt.grid(True, alpha=0.3)
plt.yticks([0, 1])

# Plot confidence probabilities
plt.subplot(2, 1, 2)
plt.plot(x_points, probs, 'r-', linewidth=2)
plt.xlabel('Distance (m)')
plt.ylabel('Confidence')
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
#histogram of preds
plt.figure(figsize=(10, 6))
plt.hist(preds, bins=np.arange(-0.5, 20, 1), alpha=0.7, color='blue')
plt.xticks(range(21))
plt.xlabel('Predicted Class')
plt.ylabel('Frequency')
plt.title('Histogram of Predictions')
#print fresquency on top of bars
for i in range(len(np.unique(preds))):
    plt.text(i, np.bincount(preds)[i], str(np.bincount(preds)[i]), ha='center', va='bottom')

plt.grid(axis='y', alpha=0.3)
plt.show()



In [ ]:
def filter_consecutive(indices):
    if not indices:
        return []
    filtered = []
    group = [indices[0]]
    for i in range(1, len(indices)):
        if indices[i] == indices[i-1] + 1:
            group.append(indices[i])
        else:
            # End of a consecutive group
            filtered.append(group[len(group)//2])
            group = [indices[i]]
    # Add the last group
    filtered.append(group[len(group)//2])
    return filtered



In [ ]:
positive_indices = [i for i, (p, prob) in enumerate(zip(preds, probs)) if p != 0 and prob > 0.9]
filtered_positive_indices = filter_consecutive(positive_indices)
print(len(filtered_positive_indices), "positive indices found")

In [6]:
model.to(device)
positive_indices = [i for i, (p, prob) in enumerate(zip(preds, probs)) if p != 0 and prob > 0.9]
filtered_positive_indices = filter_consecutive(positive_indices)
print(len(filtered_positive_indices), "positive indices found")
positive_VD = [((img_start + img_end) / 2) for (img_start, img_end) in np.array(ranges)[filtered_positive_indices]]
positive_probs = [probs[i] for i in filtered_positive_indices]
positive_preds = [preds[i] for i in filtered_positive_indices]
out_dir = os.path.join(root_dir, "test_21class/pos")
os.makedirs(out_dir, exist_ok=True)
model.eval()
with torch.no_grad():
    for vd in positive_VD:

        img_std = gen_image(vd, show_im=True, normalize="std")[:20, :]
        img_patch = gen_image(vd, show_im=False, normalize="patch")[:20, :]
        img_patch_trend = seasonal_decompose(img_patch.T, model='additive', period=30, extrapolate_trend=2).trend.T
        img = np.stack((img_patch, img_patch_trend, img_std), axis=0)  # shape: (3, H, W)
        img = transform(img).unsqueeze(0).to(device)
        #np.save(os.path.join(out_dir, f"img_{vd:.2f}"), img)

        model.to(device)
        model.eval()
        raw_outputs = model(img.to(device))
        pred = raw_outputs.argmax().cpu().item()
        prob = raw_outputs.sigmoid().max().cpu()
        print("VD:", vd, "Prediction:", pred, "Probability:", prob)

        print("Predicted indices:", preds_list[vd])




NameError: name 'model' is not defined

## upload bookmarks

In [ ]:
from __future__ import annotations

from ilipy import Session
from ilipy.channeldata import ChannelDataReader, ImageProfile
from ilipy.features import Bookmarks

from ilipyutils.ml_features.base import (
    AnomalyStatus,
    get_ml_models_info_list,
)
from ilipyutils.ml_features.insert import FeatureInsert
from ilipy import ClipTypes, TrackIndex, ViewDistance

session = Session(environment="research")
bookmarks_interface = Bookmarks(connector=session.database_connector)

feature_insert = FeatureInsert(session=session, bookmarks_interface=bookmarks_interface)

In [ ]:

anomalies = bookmarks_interface.get_anomalies(inspectionId="09WMV85VAMM")
dent_anomalies = [a for a in anomalies if "Dent-Detection-v1" in a.tags]
len(dent_anomalies)


In [ ]:
_ = [bookmarks_interface.delete_anomaly(a.feature.anomaly_feature_id) for a in dent_anomalies]

In [ ]:

inspection_id = "09WMV85VAMM"
model_info = next(
    model
    for model in get_ml_models_info_list()
    if model.model_name == "Dent-Detection-v1"
)
model_info



In [ ]:

session = Session(environment="research")
session.set_active_inspection(inspection_id)
dist_corr = DistanceCorrelation(session)

In [ ]:

from ili_custom_data.load_model import ModelDataLoader

latest_model = ModelDataLoader.get_latest_model()
model_instance = latest_model(
    ml_confidence_score =0.85
)
print(model_instance)

In [ ]:

session.set_active_inspection(inspection_id)
clips = session.get_clips_by_type(ClipTypes.ChannelData)

In [ ]:

def get_decimal_range(value):
    """
    For a number like 0.56, return (0.5, 0.6)
    For a number like 0.78, return (0.7, 0.8)
    
    Args:
        value (float): A decimal value 
        
    Returns:
        tuple: The lower and upper bounds of the decimal range
    """
    # Get the first decimal place as the lower bound
    lower = int(value * 10) / 10
    # The upper bound is just 0.1 more
    upper = round(lower + 0.1, 1)
    
    return lower, upper

In [ ]:


vd = 121.9
# 5,6

profile = ImageProfile.ZeroAngle 
track_nums = [6]

min_track_num = min(track_nums)
max_track_num = max(track_nums)+1

bookmark_conf_score = prob
lower_bound, upper_bound = get_decimal_range(bookmark_conf_score)
extra_tags = {f"threshold_bin-{lower_bound}-{upper_bound}"}

start_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(min_track_num), ViewDistance(vd-(patch_length/2))).value
end_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(max_track_num), ViewDistance(vd+(patch_length/2))).value


model_instance = latest_model(
ml_confidence_score=bookmark_conf_score,)

anomaly_info = feature_insert.insert_ml_pred(
inspection_id=inspection_id,
tlbr_track_indices=(min_track_num, max_track_num),
tlbr_odometer_ticks=(start_frame_odo_ticks, end_frame_odo_ticks),
tlbr_scan_angles=(np.deg2rad(-15),np.deg2rad(15)),
model_info=model_info,
anomaly_status=AnomalyStatus.UNKNOWN,
image_profile=profile,
extra_tags=extra_tags,
ili_custom_data=model_instance
)

In [ ]:
from ilipy.bookmarks import Bookmarks

In [ ]:
dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(min_track_num), ViewDistance(vd-patch_length/2)).value,  dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(min_track_num), ViewDistance(vd+patch_length/2)).value

In [ ]:
len(positive_probs)

In [ ]:

for i, (vd, prob, pred) in enumerate(zip(positive_VD, positive_probs, positive_preds)):


    profile = ImageProfile.ZeroAngle 
    track_nums = preds_list[vd]
    #remove zero track and reduce one
    track_nums = [track_num-1 for track_num in track_nums if track_num != 0]
    min_track_num = min(track_nums)
    max_track_num = max(track_nums)+1

    bookmark_conf_score = prob
    lower_bound, upper_bound = get_decimal_range(bookmark_conf_score)
    extra_tags = {f"threshold_bin-{lower_bound}-{upper_bound}"}

    start_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(min_track_num), ViewDistance(vd-patch_length/2)).value
    end_frame_odo_ticks = dist_corr.get_odometer_ticks_from_view_distance(TrackIndex(max_track_num), ViewDistance(vd+patch_length/2)).value
    model_instance = latest_model(
    ml_confidence_score=bookmark_conf_score,)

    anomaly_info = feature_insert.insert_ml_pred(
    inspection_id=inspection_id,
    tlbr_track_indices=(min_track_num, max_track_num),
    tlbr_odometer_ticks=(start_frame_odo_ticks, end_frame_odo_ticks),
    tlbr_scan_angles=(np.deg2rad(-15),np.deg2rad(15)),
    model_info=model_info,
    anomaly_status=AnomalyStatus.UNKNOWN,
    image_profile=profile,
    extra_tags=extra_tags,
    ili_custom_data=model_instance
    )
	

In [ ]:
import os
import numpy as np
import torch

# --- assume you already have these from the big batch pass ---
# preds        : List[int]   predicted 22-track index (0..21) per sample
# probs        : List[float] global prob per sample (max over 22 tracks)
# ranges       : List[Tuple[float,float]]
# preds_list   : Dict[vd -> List[int]] high-prob tracks for that window (optional)
# root_dir, device, model, normalizer, gen_image, filter_consecutive available

# auto-match deltas with the trained model
auto_use_deltas = (getattr(model, "per_track_in_ch", 1) == 2)

# pick positive candidates exactly like your original logic
positive_indices = [i for i, (p, prob) in enumerate(zip(preds, probs)) if p != 0 and prob > 0.9]
filtered_positive_indices = filter_consecutive(positive_indices)
print(len(filtered_positive_indices), "positive indices found")

positive_VD = [((img_start + img_end) / 2.0) for (img_start, img_end) in np.array(ranges)[filtered_positive_indices]]
positive_probs = [probs[i] for i in filtered_positive_indices]
positive_preds = [preds[i] for i in filtered_positive_indices]

out_dir = os.path.join(root_dir, "test_tcn/pos")
os.makedirs(out_dir, exist_ok=True)

# --- helpers ---
def run_slice(x20: torch.Tensor) -> torch.Tensor:
    """
    x20: (B, 20, T) on same device
    -> returns per-track probabilities (B, 20)
    """
    X_feat = build_inputs_linear(x20, use_deltas=auto_use_deltas)  # (B, 20|40, T)
    exp_cin = 20 * getattr(model, "per_track_in_ch", 1)
    assert X_feat.shape[1] == exp_cin, f"Cin={X_feat.shape[1]} but model expects {exp_cin}"
    X_feat = normalizer(X_feat)
    logits = model.forward_with_features(X_feat)   # (B, 20)
    return torch.sigmoid(logits)                   # (B, 20)

# Optional: disable deterministic algos during eval to avoid cuBLAS determinism error
was_det = torch.are_deterministic_algorithms_enabled()
if was_det:
    torch.use_deterministic_algorithms(False)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

model.eval().to(device)
if hasattr(normalizer, "to"):
    normalizer.to(device)

with torch.no_grad():
    for vd in positive_VD:
        # ---- build 22-track time-series window for this VD ----
        # gen_image should return (22, T) float array for your time-series
        arr_22 = gen_image(vd, show_im=True, normalize="patch")[:22, :]  # (22, T)
        arr_22 = np.nan_to_num(arr_22, nan=0.0, posinf=0.0, neginf=0.0)

        x_22 = torch.from_numpy(arr_22).float().unsqueeze(0).to(device)   # (1, 22, T)

        # two 20-track slices:
        x_a = x_22[:, :20, :]   # -> global tracks 0..19
        x_b = x_22[:,  2:, :]   # -> global tracks 2..21

        p_a = run_slice(x_a).squeeze(0)  # (20,)
        p_b = run_slice(x_b).squeeze(0)  # (20,)

        # fuse into 22-track prob vector
        probs_22 = torch.empty(22, device=device)
        probs_22[0:20] = p_a
        # average overlap 2..19 (use max if you prefer):
        probs_22[2:20] = 0.5 * (probs_22[2:20] + p_b[0:18])
        probs_22[20:22] = p_b[18:20]

        p_global = probs_22.max().item()
        pred_track = int(torch.argmax(probs_22).item())

        print(f"VD: {vd:.3f} | Predicted track (0..21): {pred_track} | Global prob: {p_global:.4f}")

        # optional: save the raw 22xT input and 22-track probs for this VD
        np.save(os.path.join(out_dir, f"window_{vd:.3f}_x22.npy"), arr_22)
        np.save(os.path.join(out_dir, f"window_{vd:.3f}_probs22.npy"), probs_22.detach().cpu().numpy())

        # if you built preds_list earlier, show the high-prob tracks for this VD (if any)
        if vd in preds_list:
            print("High-prob tracks (from pass 1):", preds_list[vd])
        else:
            print("High-prob tracks: (none recorded)")

# restore determinism flags if changed
if was_det:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
